# 04 -- Cross-Attention Conditioning (Centrepiece)

**Author**: Tamer Atesyakar

This is the load-bearing notebook. We construct a small `ConditioningProjector` that maps a 7-dim `AdaptationVector` into conditioning-token space, plug it into a minimal pre-norm transformer decoder block via *cross-attention*, and run the **conditioning-sensitivity test**: given the same prompt, we measure that two different adaptation vectors induce next-token distributions with *measurably* large Kullback-Leibler divergence. This is the necessary condition for the adaptation pipeline to actually influence generation.

**Citations**
- Vaswani et al. (2017). *Attention is All You Need.* NeurIPS.
- Xiong et al. (2020). *On Layer Normalization in the Transformer Architecture.* ICML.
- Kingma & Ba (2015). *Adam: A Method for Stochastic Optimization.* ICLR.

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    TORCH_OK = True
except ImportError as e:
    TORCH_OK = False
    print(f'torch unavailable: {e}')

C_COLD = '#0f3460'
C_HOT  = '#e94560'
plt.rcParams.update({'axes.grid': True, 'grid.alpha': 0.25})

## 4.1 The AdaptationVector

Seven scalar dials in [0, 1], each describing how to shape a response:
`verbosity, formality, directness, warmth, cognitive_accessibility, emotional_tone, technical_depth`.

In [ ]:
ADAPT_DIMS = ['verbosity','formality','directness','warmth',
              'cognitive_accessibility','emotional_tone','technical_depth']

def adapt(v=0.5, f=0.5, d=0.5, w=0.5, a=0.5, e=0.5, t=0.5):
    return torch.tensor([[v, f, d, w, a, e, t]], dtype=torch.float32)

if TORCH_OK:
    av_a = adapt(v=0.25, d=0.85, a=0.9)   # low cognitive load
    av_b = adapt(v=0.9,  w=0.95, e=0.9)   # high warmth, chatty
    print(dict(zip(ADAPT_DIMS, av_a[0].tolist())))
    print(dict(zip(ADAPT_DIMS, av_b[0].tolist())))

## 4.2 ConditioningProjector

Takes the 7-dim adaptation vector and projects it to `K` conditioning tokens of dimension `d_model` using a small MLP that emits `K * d_model` values. The K tokens are treated as a sequence the decoder can cross-attend over.

In [ ]:
if TORCH_OK:
    class ConditioningProjector(nn.Module):
        def __init__(self, d_adapt=7, d_model=32, k_tokens=4):
            super().__init__()
            self.k = k_tokens
            self.d = d_model
            self.mlp = nn.Sequential(
                nn.Linear(d_adapt, 64),
                nn.GELU(),
                nn.Linear(64, k_tokens * d_model),
            )
            self.pos = nn.Parameter(torch.randn(1, k_tokens, d_model) * 0.02)

        def forward(self, a):
            # a: (B, d_adapt) -> (B, k, d_model)
            y = self.mlp(a).view(a.shape[0], self.k, self.d)
            return y + self.pos

    proj = ConditioningProjector()
    toks = proj(av_a)
    print('conditioning tokens shape:', tuple(toks.shape))

## 4.3 Minimal pre-norm transformer block with cross-attention

In [ ]:
if TORCH_OK:
    class TinyXAttnBlock(nn.Module):
        def __init__(self, d_model=32, n_heads=4, d_ff=64, dropout=0.0):
            super().__init__()
            self.ln1 = nn.LayerNorm(d_model)
            self.self_attn = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
            self.ln2 = nn.LayerNorm(d_model)
            self.cross_attn = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
            self.ln3 = nn.LayerNorm(d_model)
            self.ff = nn.Sequential(
                nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model),
            )
            self.drop = nn.Dropout(dropout)
            self.last_attn_weights = None

        def forward(self, x, cond):
            # Pre-norm (Xiong 2020) -- more stable than post-norm for small models
            h = self.ln1(x)
            sa, _ = self.self_attn(h, h, h, need_weights=False)
            x = x + self.drop(sa)
            h = self.ln2(x)
            ca, w = self.cross_attn(h, cond, cond, need_weights=True, average_attn_weights=True)
            self.last_attn_weights = w.detach()     # (B, T_q, T_k)
            x = x + self.drop(ca)
            x = x + self.drop(self.ff(self.ln3(x)))
            return x

    class TinyLM(nn.Module):
        def __init__(self, vocab=64, d_model=32, n_blocks=4, k_cond=4):
            super().__init__()
            self.embed = nn.Embedding(vocab, d_model)
            self.pos   = nn.Parameter(torch.randn(1, 32, d_model) * 0.02)
            self.proj  = ConditioningProjector(d_adapt=7, d_model=d_model, k_tokens=k_cond)
            self.blocks = nn.ModuleList(
                [TinyXAttnBlock(d_model=d_model) for _ in range(n_blocks)]
            )
            self.ln_f  = nn.LayerNorm(d_model)
            self.head  = nn.Linear(d_model, vocab, bias=False)

        def forward(self, tokens, adapt_vec):
            B, T = tokens.shape
            x = self.embed(tokens) + self.pos[:, :T]
            cond = self.proj(adapt_vec)
            for blk in self.blocks:
                x = blk(x, cond)
            return self.head(self.ln_f(x))

    m = TinyLM(vocab=64)
    n = sum(p.numel() for p in m.parameters())
    print(f'TinyLM parameters: {n:,}')

## 4.4 Train on a toy objective that exposes the conditioning

Teaching signal: given a prompt `[BOS, 1, 2, 3]`, the target last-token is either `HIGH` or `LOW` depending on `adapt[0]` (verbosity). If the conditioning channel is functional, the model learns to use it; if not, it cannot solve the task.

In [ ]:
if TORCH_OK:
    VOCAB = 64; BOS = 1; HIGH = 40; LOW = 10
    torch.manual_seed(0)
    lm  = TinyLM(vocab=VOCAB)
    opt = torch.optim.Adam(lm.parameters(), lr=3e-3)

    def make_batch(n=32):
        toks = torch.tensor([[BOS, 2, 3, 4]] * n)
        a = torch.rand(n, 7)
        # Label depends on verbosity > 0.5
        y = torch.where(a[:, 0] > 0.5, torch.full((n,), HIGH), torch.full((n,), LOW))
        return toks, a, y

    losses = []
    for step in range(300):
        toks, a, y = make_batch()
        logits = lm(toks, a)                # (B, T, V)
        loss = F.cross_entropy(logits[:, -1], y)
        opt.zero_grad(); loss.backward(); opt.step()
        losses.append(loss.item())
    print(f'final CE: {losses[-1]:.3f}')

    fig, ax = plt.subplots(figsize=(8, 3))
    ax.plot(losses, color=C_HOT); ax.set_title('Training loss')
    ax.set_xlabel('step'); ax.set_ylabel('CE')
    plt.tight_layout(); plt.show()

## 4.5 The conditioning-sensitivity test

For the same prompt, we compute `p(next | adapt=A)` and `p(next | adapt=B)` and measure their KL divergence. A working conditioning pathway must yield a *materially large* KL -- otherwise the adaptation vector has no effect on the output distribution.

In [ ]:
if TORCH_OK:
    lm.eval()
    with torch.no_grad():
        toks = torch.tensor([[BOS, 2, 3, 4]])
        p_a = F.softmax(lm(toks, av_a)[0, -1], dim=-1)
        p_b = F.softmax(lm(toks, av_b)[0, -1], dim=-1)
        kl = torch.sum(p_a * (torch.log(p_a + 1e-12) - torch.log(p_b + 1e-12))).item()
    print(f'KL(p_A || p_B) = {kl:.4f} nats  ({kl/math.log(2):.4f} bits)')
    print('(For an ineffective conditioning pathway this would be ~0.)')
    print(f'argmax under A: token {int(p_a.argmax())}   expected {HIGH} (verbosity>0.5? {av_a[0,0].item()>0.5})')
    print(f'argmax under B: token {int(p_b.argmax())}   expected {HIGH} (verbosity>0.5? {av_b[0,0].item()>0.5})')

## 4.6 Visualising cross-attention weights

In [ ]:
if TORCH_OK:
    with torch.no_grad():
        _ = lm(toks, av_a)
    rows = []
    for blk in lm.blocks:
        w = blk.last_attn_weights[0, -1].cpu().numpy()   # last query over K keys
        rows.append(w)
    mat = np.stack(rows)

    fig, ax = plt.subplots(figsize=(5.2, 4))
    from matplotlib.colors import LinearSegmentedColormap
    cmap = LinearSegmentedColormap.from_list('i3', [C_COLD, C_HOT])
    im = ax.imshow(mat, cmap=cmap, aspect='auto', vmin=0, vmax=mat.max())
    ax.set_xlabel('conditioning token')
    ax.set_ylabel('transformer block')
    ax.set_title('Cross-attention: last generated token -> conditioning tokens')
    for r in range(mat.shape[0]):
        for c in range(mat.shape[1]):
            ax.text(c, r, f'{mat[r,c]:.2f}', ha='center', va='center',
                    fontsize=9, color='white' if mat[r,c] > mat.max()/2 else '#333')
    fig.colorbar(im, ax=ax, shrink=0.8)
    plt.tight_layout(); plt.show()

## 4.7 KL sweep across the adaptation simplex

We sweep one dimension (verbosity) while holding the rest fixed and plot the KL of the output distribution versus a neutral baseline.

In [ ]:
if TORCH_OK:
    baseline = adapt()              # all 0.5
    xs = np.linspace(0.0, 1.0, 21)
    kls = []
    with torch.no_grad():
        p_base = F.softmax(lm(toks, baseline)[0, -1], dim=-1)
        for v in xs:
            a = adapt(v=float(v))
            p = F.softmax(lm(toks, a)[0, -1], dim=-1)
            k = torch.sum(p * (torch.log(p + 1e-12) - torch.log(p_base + 1e-12))).item()
            kls.append(k)
    fig, ax = plt.subplots(figsize=(7, 3.2))
    ax.plot(xs, kls, color=C_HOT, lw=2, marker='o')
    ax.axvline(0.5, color='#555', ls='--', lw=0.7, label='baseline')
    ax.set_xlabel('verbosity'); ax.set_ylabel('KL(p || p_baseline) [nats]')
    ax.set_title('Output distribution sensitivity to verbosity')
    ax.legend(); plt.tight_layout(); plt.show()

## 4.8 Why this matters

The conditioning-sensitivity test is the single most diagnostic check for the I3 architecture: without it, the entire adaptation pipeline could be a *placebo*. By measuring a non-trivial KL between outputs conditioned on different adaptation vectors, we empirically verify that cross-attention is actually routing information from the adaptation pathway into the generation step (Vaswani 2017, Xiong 2020).